In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# =============================================================================
# INPUT FILES
# =============================================================================

SENSITIVITY_CSV = Path("data") / "sensitivity.csv"
COPING_CSV = Path("data") / "coping_capacity.csv"
OUTPUT_CSV = Path("data") / "vulnerability.csv"

sens = pd.read_csv(SENSITIVITY_CSV)
cc = pd.read_csv(COPING_CSV)

print("Sensitivity shape:", sens.shape)
print("Coping shape:", cc.shape)

# =============================================================================
# MERGE DATA
# =============================================================================

df = sens.merge(
    cc,
    on=["district", "timeperiod"],
    how="inner"
)

# =============================================================================
# INVERT COPING CAPACITY
# =============================================================================
# 1 ↔ 5, 2 ↔ 4, 3 ↔ 3

df["inv_coping"] = 6 - df["coping_capacity"]

# =============================================================================
# VULNERABILITY RAW SCORE
# =============================================================================

df["vulnerability_raw"] = (
    df["sensitivity_class"] + df["inv_coping"]
)

# =============================================================================
# Z-SCORE FUNCTION
# =============================================================================

def zscore(x):
    std = x.std(ddof=0)
    if std == 0:
        return pd.Series(0, index=x.index)
    return (x - x.mean()) / std


def classify(z):
    if z <= -1.5:
        return 1
    elif z <= -0.5:
        return 2
    elif z <= 0.5:
        return 3
    elif z <= 1.5:
        return 4
    else:
        return 5


# =============================================================================
# MONTH-WISE Z-SCORE (ACROSS DISTRICTS)
# =============================================================================

df["vuln_z"] = (
    df.groupby("timeperiod")["vulnerability_raw"]
    .transform(zscore)
)

# =============================================================================
# FINAL BINNING
# =============================================================================

df["vulnerability_class"] = df["vuln_z"].apply(classify)

# =============================================================================
# OUTPUT
# =============================================================================

output_cols = [
    "district",
    "timeperiod",
    "sensitivity_class",
    "coping_capacity",
    "inv_coping",
    "vulnerability_raw",
    "vuln_z",
    "vulnerability_class"
]

df[output_cols].to_csv(OUTPUT_CSV, index=False)

print("Saved:", OUTPUT_CSV)

# =============================================================================
# CHECKS
# =============================================================================

print("\nVulnerability distribution:")
print(df["vulnerability_class"].value_counts().sort_index())

print("\nPreview:")
print(df.head())

Sensitivity shape: (690, 20)
Coping shape: (690, 14)
Saved: data/vulnerability.csv

Vulnerability distribution:
vulnerability_class
1     69
2    184
3     92
4    345
Name: count, dtype: int64

Preview:
  district timeperiod   aged_pop  young_pop  no_sanitation  nco_5_9  pct_ncd  \
0   Anugul    2023_01  84749.045  90931.373      13.656656     79.0     20.0   
1   Anugul    2023_02  84749.045  90931.373      13.656656     79.0     20.0   
2   Anugul    2023_03  84749.045  90931.373      13.656656     79.0     20.0   
3   Anugul    2023_04  84749.045  90931.373      13.656656     79.0     20.0   
4   Anugul    2023_05  84749.045  90931.373      13.656656     79.0     20.0   

   aged_pop_z  young_pop_z  no_sanitation_z  ...  health_centers_bin  \
0   -0.426815    -0.440308        -0.051004  ...                   3   
1   -0.426815    -0.440308        -0.051004  ...                   3   
2   -0.426815    -0.440308        -0.051004  ...                   3   
3   -0.426815    -0.440308 